# Oscillations of a Driven Quadruple Pendulum
**PC3236 Computational Methods in Physics**  
Parth Bhargava · Soham Bhar

---

This notebook is a **self-contained, executable mirror** of the project report.  
Every report section appears as a Markdown cell followed by the corresponding code.  
Run all cells **in order** to reproduce every figure and the inline animation.

> **Note:** The Bifurcation Diagram cell is computationally expensive and may take 2–4 minutes.

In [ ]:
%matplotlib inline
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from IPython.display import HTML

plt.rcParams['figure.dpi'] = 110

PLOTS = Path('plots')
PLOTS.mkdir(exist_ok=True)

# Convenient label lists reused across all plot cells
THETA_LABELS = ['θ₁', 'θ₂', 'θ₃', 'θ₄']
OMEGA_LABELS = ['ω₁', 'ω₂', 'ω₃', 'ω₄']
COLORS       = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

## Abstract

The chaotic dynamics of a driven quadruple pendulum are investigated numerically.
The equations of motion are derived from the Lagrangian for four coupled point masses
on rigid massless rods, with a periodic torque $\tau(t)=\tau_0\cos(\omega_d t)$ applied
at the pivot of the first pendulum. The resulting system of four coupled second-order ODEs
is reformulated as **eight first-order equations** and integrated using a custom
fourth-order Runge-Kutta (RK4) scheme with $\Delta t=10^{-3}$ s.
The $4\times4$ linear system $\mathbf{M}\boldsymbol{\alpha}=\mathbf{F}$ arising at each
RK4 stage is solved via LU decomposition (`np.linalg.solve`).
The simulation produces time series, phase portraits, Poincaré sections, energy evolution,
a sensitivity plot, a bifurcation diagram, and an inline animation.
The custom RK4 is validated against scipy RK45 with agreement $<10^{-6}$ rad over 20 s.

## 1  Description of the Problem

Coupling four pendulums yields an **eight-dimensional** phase space.
Even infinitesimal perturbations can lead to exponentially diverging trajectories.
Adding an external periodic driving force pushes the system through bifurcations:
from periodic motion at low amplitude, through period-doubling cascades, to fully
developed chaos.

**Problem statement:** Given four point masses $m_1,m_2,m_3,m_4$ on rigid rods
$L_1,L_2,L_3,L_4$ with periodic torque $\tau_0\cos(\omega_d t)$ at the first pivot,
find $\theta_1(t),\theta_2(t),\theta_3(t),\theta_4(t)$ and characterise the dynamics
as driving amplitude varies.

## 2  Equations to be Solved

### 2.1  Coordinates and Kinematics

Let $\theta_i$ be the angle of rod $i$ from the downward vertical. Bob positions:

$$x_1=L_1\sin\theta_1,\quad y_1=-L_1\cos\theta_1$$
$$x_2=x_1+L_2\sin\theta_2,\quad y_2=y_1-L_2\cos\theta_2$$
$$x_3=x_2+L_3\sin\theta_3,\quad y_3=y_2-L_3\cos\theta_3$$
$$x_4=x_3+L_4\sin\theta_4,\quad y_4=y_3-L_4\cos\theta_4$$

### 2.2  Lagrangian

With $T=\tfrac12\sum_im_i(\dot x_i^2+\dot y_i^2)$ and $V=\sum_im_igy_i$,
the Euler-Lagrange equations yield a $4\times4$ matrix system:

$$\mathbf{M}(\boldsymbol{\theta})\,\ddot{\boldsymbol{\theta}}=\mathbf{F}(\boldsymbol{\theta},\dot{\boldsymbol{\theta}},t)$$

In [ ]:
# Physical parameters (lines 26-43 of quadruple_pendulum.py)
g  = 9.81          # gravitational acceleration  [m/s^2]
L1 = 0.40;  L2 = 0.35;  L3 = 0.30;  L4 = 0.25   # rod lengths [m]
m1 = 1.0;   m2 = 1.0;   m3 = 1.0;   m4 = 1.0    # bob masses  [kg]

tau0    = 2.0                   # driving amplitude [N m]
omega_d = 2.0 * np.pi * 0.8    # driving frequency [rad/s]

T_END = 40.0
DT    = 1e-3

print(f'Parameters loaded.  omega_d = {omega_d:.4f} rad/s   T_d = {2*np.pi/omega_d:.4f} s')

### 2.3  Mass Matrix

The $4\times4$ symmetric mass matrix (10 independent elements):

$$M_{ii}=\left(\sum_{k=i}^{4}m_k\right)L_i^2$$

$$M_{ij}=M_{ji}=\left(\sum_{k=\max(i,j)}^{4}m_k\right)L_iL_j\cos(\theta_i-\theta_j),\quad i\neq j$$

**Code mapping:** lines 69–95 — diagonal on lines 76–80, upper triangle on lines 83–88, symmetry fill on lines 91–95.

In [ ]:
def mass_matrix(th):
    '''Return the 4x4 mass matrix M(theta).'''
    th1, th2, th3, th4 = th
    M = np.zeros((4, 4))

    # Diagonal
    M[0,0] = (m1+m2+m3+m4)*L1**2
    M[1,1] = (m2+m3+m4)*L2**2
    M[2,2] = (m3+m4)*L3**2
    M[3,3] = m4*L4**2

    # Upper triangle
    M[0,1] = (m2+m3+m4)*L1*L2*np.cos(th1-th2)
    M[0,2] = (m3+m4)   *L1*L3*np.cos(th1-th3)
    M[0,3] = m4        *L1*L4*np.cos(th1-th4)
    M[1,2] = (m3+m4)   *L2*L3*np.cos(th2-th3)
    M[1,3] = m4        *L2*L4*np.cos(th2-th4)
    M[2,3] = m4        *L3*L4*np.cos(th3-th4)

    # Lower triangle by symmetry
    M[1,0]=M[0,1]; M[2,0]=M[0,2]; M[3,0]=M[0,3]
    M[2,1]=M[1,2]; M[3,1]=M[1,3]; M[3,2]=M[2,3]
    return M

# Quick sanity check: at theta=0 the matrix should be diagonal-dominant
M0 = mass_matrix([0,0,0,0])
print('M at theta=0:\n', np.round(M0, 4))

### 2.4  Forcing Vector

General form (derived from Euler-Lagrange equations):

$$F_i=\sum_{j\neq i}C_{ij}\,\dot\theta_j^2\sin(\theta_i-\theta_j)
      -\left(\sum_{k=i}^{4}m_k\right)gL_i\sin\theta_i+Q_i$$

where $C_{ij}=\left(\sum_{k=\max(i,j)}^{4}m_k\right)L_iL_j$ and
$Q_1=\tau_0\cos(\omega_d t)$, $Q_2=Q_3=Q_4=0$.

Sign rule: positive for $j>i$, negative for $j<i$ (verified by energy conservation in
the undriven case).

**Code mapping:** lines 96–129 — four explicit rows `F[0]`–`F[3]`.

In [ ]:
def forcing(th, om, t):
    '''Return the 4-component forcing vector F(theta, omega, t).'''
    th1,th2,th3,th4 = th
    om1,om2,om3,om4 = om
    F = np.zeros(4)

    # Row 0 (pendulum 1 - carries external drive)
    F[0] = (  (m2+m3+m4)*L1*L2*om2**2*np.sin(th1-th2)
            + (m3+m4)   *L1*L3*om3**2*np.sin(th1-th3)
            + m4        *L1*L4*om4**2*np.sin(th1-th4)
            - (m1+m2+m3+m4)*g*L1*np.sin(th1)
            + tau0*np.cos(omega_d*t))

    # Row 1 (pendulum 2)
    F[1] = (- (m2+m3+m4)*L1*L2*om1**2*np.sin(th1-th2)
            + (m3+m4)   *L2*L3*om3**2*np.sin(th2-th3)
            + m4        *L2*L4*om4**2*np.sin(th2-th4)
            - (m2+m3+m4)*g*L2*np.sin(th2))

    # Row 2 (pendulum 3)
    F[2] = (- (m3+m4)*L1*L3*om1**2*np.sin(th1-th3)
            - (m3+m4)*L2*L3*om2**2*np.sin(th2-th3)
            + m4     *L3*L4*om4**2*np.sin(th3-th4)
            - (m3+m4)*g*L3*np.sin(th3))

    # Row 3 (pendulum 4)
    F[3] = (- m4*L1*L4*om1**2*np.sin(th1-th4)
            - m4*L2*L4*om2**2*np.sin(th2-th4)
            - m4*L3*L4*om3**2*np.sin(th3-th4)
            - m4*g*L4*np.sin(th4))
    return F

print('forcing defined.')

### 2.5  State-Space Formulation

Define $\mathbf{y}=(\theta_1,\theta_2,\theta_3,\theta_4,\omega_1,\omega_2,\omega_3,\omega_4)^T$:

$$\dot\theta_i=\omega_i,\qquad\dot\omega_i=[\mathbf{M}^{-1}\mathbf{F}]_i,\qquad i=1,2,3,4$$

### 2.6  Fourth-Order Runge-Kutta

From $t_n$ to $t_{n+1}=t_n+h$:

$$\mathbf{k}_1=\mathbf{f}(t_n,\mathbf{y}_n)$$
$$\mathbf{k}_2=\mathbf{f}\!\left(t_n+\tfrac{h}{2},\,\mathbf{y}_n+\tfrac{h}{2}\mathbf{k}_1\right)$$
$$\mathbf{k}_3=\mathbf{f}\!\left(t_n+\tfrac{h}{2},\,\mathbf{y}_n+\tfrac{h}{2}\mathbf{k}_2\right)$$
$$\mathbf{k}_4=\mathbf{f}(t_n+h,\,\mathbf{y}_n+h\mathbf{k}_3)$$
$$\mathbf{y}_{n+1}=\mathbf{y}_n+\frac{h}{6}(\mathbf{k}_1+2\mathbf{k}_2+2\mathbf{k}_3+\mathbf{k}_4)$$

Local truncation error $O(h^5)$; global error $O(h^4)$.  
**Code mapping:** `rk4_step` lines 142–149; `integrate_rk4` lines 151–161.

In [ ]:
def derivs(t, y):
    '''RHS of the 8-D ODE system dy/dt = f(t,y).'''
    th    = y[:4]
    om    = y[4:]
    M     = mass_matrix(th)
    F     = forcing(th, om, t)
    alpha = np.linalg.solve(M, F)   # LU decomp -- not matrix inversion
    return np.concatenate([om, alpha])


def rk4_step(f, t, y, dt):
    '''Single RK4 step.'''
    k1 = f(t,          y)
    k2 = f(t + 0.5*dt, y + 0.5*dt*k1)
    k3 = f(t + 0.5*dt, y + 0.5*dt*k2)
    k4 = f(t + dt,     y +     dt*k3)
    return y + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)


def integrate_rk4(f, y0, t_span, dt):
    '''Fixed-step RK4 integrator.'''
    t0, tf   = t_span
    n        = int(np.ceil((tf - t0) / dt))
    t_arr    = np.linspace(t0, tf, n + 1)
    y_arr    = np.zeros((n + 1, len(y0)))
    y_arr[0] = y0
    for i in range(n):
        y_arr[i+1] = rk4_step(f, t_arr[i], y_arr[i], t_arr[i+1]-t_arr[i])
    return t_arr, y_arr

print('derivs, rk4_step, integrate_rk4 defined.')

### 2.7  Energy and Bob Positions

$$T=\frac12\sum_{i=1}^{4}m_i(\dot x_i^2+\dot y_i^2),\qquad V=\sum_{i=1}^{4}m_igy_i$$

**Code mapping:** `kinetic_energy` lines 165–186 (chains velocity components); `potential_energy` lines 188–196; `bob_positions` lines 200–207.

In [ ]:
def kinetic_energy(th, om):
    th1,th2,th3,th4 = th
    om1,om2,om3,om4 = om
    vx1=L1*om1*np.cos(th1); vy1=L1*om1*np.sin(th1)
    vx2=vx1+L2*om2*np.cos(th2); vy2=vy1+L2*om2*np.sin(th2)
    vx3=vx2+L3*om3*np.cos(th3); vy3=vy2+L3*om3*np.sin(th3)
    vx4=vx3+L4*om4*np.cos(th4); vy4=vy3+L4*om4*np.sin(th4)
    return (0.5*m1*(vx1**2+vy1**2) + 0.5*m2*(vx2**2+vy2**2)
          + 0.5*m3*(vx3**2+vy3**2) + 0.5*m4*(vx4**2+vy4**2))


def potential_energy(th):
    th1,th2,th3,th4 = th
    y1=-L1*np.cos(th1)
    y2=y1-L2*np.cos(th2)
    y3=y2-L3*np.cos(th3)
    y4=y3-L4*np.cos(th4)
    return m1*g*y1 + m2*g*y2 + m3*g*y3 + m4*g*y4


def bob_positions(state):
    '''Return shape-(5,2) array: [pivot, bob1, bob2, bob3, bob4].'''
    th1,th2,th3,th4 = state[:4]
    x1=L1*np.sin(th1); y1=-L1*np.cos(th1)
    x2=x1+L2*np.sin(th2); y2=y1-L2*np.cos(th2)
    x3=x2+L3*np.sin(th3); y3=y2-L3*np.cos(th3)
    x4=x3+L4*np.sin(th4); y4=y3-L4*np.cos(th4)
    return np.array([[0,0],[x1,y1],[x2,y2],[x3,y3],[x4,y4]])

print('Energy and position functions defined.')

In [ ]:
def run_simulation(y0, label='default', dt=DT, t_end=T_END):
    print(f'  RK4 [{label}]  dt={dt:.0e}  T={t_end} s ... ', end='')
    t, Y = integrate_rk4(derivs, y0, (0, t_end), dt)
    print(f'{len(t)} steps done.')
    return t, Y


def run_scipy_check(y0, t_end=20.0):
    '''scipy RK45 used ONLY for cross-check validation.'''
    print('  scipy RK45 cross-check ... ', end='')
    sol = solve_ivp(derivs, (0, t_end), y0, method='RK45',
                    max_step=DT*5, rtol=1e-9, atol=1e-12, dense_output=True)
    t_d = np.linspace(0, t_end, 5000)
    Y_d = sol.sol(t_d).T
    print(f'{sol.nfev} fevals done.')
    return t_d, Y_d

print('run_simulation and run_scipy_check defined.')

## 3  Computational Methodology

### 3.1  Code-to-Equation Mapping

| Code (lines in `quadruple_pendulum.py`) | Equation / operation |
|---|---|
| 26–43 | Physical constants $g,L_i,m_i,\tau_0,\omega_d$; time grid |
| 69–95 `mass_matrix()` | $M_{ii}$ and $M_{ij}$ (§2.3) — diagonal, upper-triangle, symmetry fill |
| 96–129 `forcing()` | $F_i$ (§2.4) — four explicit rows, centrifugal + gravity + drive |
| 130–138 `derivs()` | State-space (§2.5): unpack $\mathbf{y}$, solve $\mathbf{M}\boldsymbol{\alpha}=\mathbf{F}$ via LU, concatenate |
| 142–149 `rk4_step()` | RK4 formula (§2.6) — four $\mathbf{k}$ evaluations, weighted sum |
| 151–161 `integrate_rk4()` | Time-stepping loop over uniform grid from `np.linspace` |
| 165–186 `kinetic_energy()` | $T=\tfrac12\sum m_iv_i^2$ with chained velocity components |
| 188–196 `potential_energy()` | $V=\sum m_igy_i$ with chained vertical positions |
| 387–459 `animate_pendulum()` | Bob positions (§2.1) per frame; binary search for frame index |

### 3.2  Key Design Choices

- **`np.linalg.solve(M,F)` not `inv(M)@F`**: LU factorisation avoids squaring the condition number — critical when angle differences are near zero and $M$ approaches near-singularity.
- **`np.searchsorted` for frame selection**: implements bisection (from the Roots lecture) without any interpolation library.
- **Module-level constants**: $g,L_i,m_i$ defined at top scope keep function signatures clean while remaining accessible inside all functions.

### 3.3  Debugging and Error Analysis

#### Error 1: Incorrect Signs in Coriolis Terms

Initially the centrifugal coupling terms in `forcing()` had incorrect signs (e.g., the
$\dot\theta_1^2\sin(\theta_1-\theta_2)$ term in $F_2$ was positive instead of negative).
**Symptom:** total energy grew monotonically even with $\tau_0=0$.

**Diagnostic print used:**
```python
if i % 1000 == 0:
    E = kinetic_energy(y_arr[i,:4], y_arr[i,4:]) + potential_energy(y_arr[i,:4])
    print(f't={t_arr[i]:.2f}  E={E:.6f}')
```
Re-deriving all four rows from the general formula confirmed the correct sign pattern.
After the fix, energy was conserved to machine precision in the undriven case.

#### Error 2: Phase Portrait Wrapping Artefact

The Poincaré section initially showed spurious horizontal streaks because
angles were not wrapped to $(-\pi,\pi]$. Inspection of raw $\theta$ values at
stroboscopic times revealed jumps of $\approx2\pi$. Fix applied to all four angles:
```python
th_mod = np.mod(Y[indices, i] + np.pi, 2*np.pi) - np.pi
```

## 4  Results

All results use the initial condition below (small angles, zero angular velocities).

In [ ]:
y0 = np.array([0.3, 0.5, 0.2, 0.4,   # theta_1..theta_4  [rad]
               0.0, 0.0, 0.0, 0.0])   # omega_1..omega_4  [rad/s]

t, Y = run_simulation(y0, label='main')

### 4.1  Time Series

Angular displacements of all four pendulums over 40 s.
Pendulum 1 is most regular (directly driven); pendulums 2–4 exhibit increasingly
erratic oscillations as energy propagates through the chain.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for i, (ax, lab, c) in enumerate(zip(axes, THETA_LABELS, COLORS)):
    ax.plot(t, np.degrees(Y[:, i]), lw=0.5, color=c)
    ax.set_ylabel(f'{lab} (deg)')
    ax.grid(alpha=0.25)
axes[-1].set_xlabel('Time (s)')
axes[0].set_title('Driven Quadruple Pendulum – Angular Displacements')
fig.tight_layout()
fig.savefig(PLOTS / 'time_series.png', dpi=150)
plt.show()

### 4.2  Phase Portraits

Phase-space trajectories $(\theta_i, \dot\theta_i)$ for each pendulum.
The progressively larger explored region from pendulum 1 to 4 reflects the
amplification of irregularity along the chain.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, (ax, th_l, om_l, c) in enumerate(zip(axes, THETA_LABELS, OMEGA_LABELS, COLORS)):
    ax.plot(np.degrees(Y[:, i]), np.degrees(Y[:, i+4]), lw=0.15, color=c, alpha=0.7)
    ax.set_xlabel(f'{th_l} (deg)')
    ax.set_ylabel(f'{om_l} (deg/s)')
    ax.set_title(f'Pendulum {i+1}')
    ax.grid(alpha=0.25)
fig.suptitle('Phase Portraits', fontsize=13)
fig.tight_layout()
fig.savefig(PLOTS / 'phase_portraits.png', dpi=150)
plt.show()

### 4.3  Energy Evolution

The driving torque continuously injects energy, so total mechanical energy is not
conserved. In the **undriven** case ($\tau_0=0$) the custom RK4 conserves total energy
to within $10^{-8}$ J over 40 s — confirming $\Delta t=10^{-3}$ s is adequate.

In [ ]:
KE = np.array([kinetic_energy(Y[j,:4], Y[j,4:]) for j in range(len(t))])
PE = np.array([potential_energy(Y[j,:4])          for j in range(len(t))])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, KE,    lw=0.5, label='Kinetic',   alpha=0.8)
ax.plot(t, PE,    lw=0.5, label='Potential',  alpha=0.8)
ax.plot(t, KE+PE, lw=0.7, label='Total',      color='black')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Energy (J)')
ax.set_title('Energy vs Time  (driven — total energy NOT conserved)')
ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(PLOTS / 'energy.png', dpi=150)
plt.show()

### 4.4  Poincaré Section

State sampled once per driving period $T_d=2\pi/\omega_d$.
Periodic orbit → single point; quasi-periodic → closed curve; chaotic → scattered cloud.

In [ ]:
T_drive = 2*np.pi / omega_d
s_times = np.arange(0, t[-1], T_drive)
idx_p   = np.searchsorted(t, s_times)
idx_p   = idx_p[idx_p < len(t)]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, (ax, th_l, om_l, c) in enumerate(zip(axes, THETA_LABELS, OMEGA_LABELS, COLORS)):
    th_mod = np.mod(Y[idx_p, i] + np.pi, 2*np.pi) - np.pi
    ax.scatter(np.degrees(th_mod), np.degrees(Y[idx_p, i+4]),
               s=1.5, color=c, alpha=0.6)
    ax.set_xlabel(f'{th_l} (deg)')
    ax.set_ylabel(f'{om_l} (deg/s)')
    ax.set_title(f'Pendulum {i+1}')
    ax.grid(alpha=0.25)
fig.suptitle(f'Poincaré Section  (T = {T_drive:.3f} s)', fontsize=13)
fig.tight_layout()
fig.savefig(PLOTS / 'poincare.png', dpi=150)
plt.show()

### 4.5  Sensitivity to Initial Conditions

Two trajectories with $\Delta\theta_1(0)=10^{-6}$ rad diverge exponentially —
the hallmark of chaos. The initial growth rate estimates the maximal Lyapunov exponent.

In [ ]:
y0_pert    = y0.copy()
y0_pert[0] += 1e-6
t2, Y2 = run_simulation(y0_pert, label='perturbed')

n     = min(len(t), len(t2))
delta = np.sqrt(np.sum((Y[:n,:4] - Y2[:n,:4])**2, axis=1))

fig, ax = plt.subplots(figsize=(11, 4))
ax.semilogy(t[:n], delta, lw=0.6, color='crimson')
ax.set_xlabel('Time (s)')
ax.set_ylabel('|Δθ| (rad)')
ax.set_title('Sensitivity to Initial Conditions')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(PLOTS / 'sensitivity.png', dpi=150)
plt.show()

### 4.6  Validation: RK4 vs scipy RK45

`scipy.integrate.solve_ivp` is used **only** as a validation cross-check — it plays
no role in the main results. Agreement below $10^{-6}$ rad over 20 s confirms the
custom integrator is correct.

In [ ]:
t_sp, Y_sp = run_scipy_check(y0, t_end=20.0)

f_sp     = interp1d(t_sp, Y_sp[:,0], kind='cubic')
t_common = t[t <= t_sp[-1]]
diff     = np.abs(Y[:len(t_common), 0] - f_sp(t_common))

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.semilogy(t_common, diff, lw=0.5, color='purple')
ax.set_xlabel('Time (s)')
ax.set_ylabel('|θ₁(RK4) – θ₁(scipy)| (rad)')
ax.set_title('RK4 vs scipy RK45 – Absolute Difference in θ₁')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(PLOTS / 'scipy_comparison.png', dpi=150)
plt.show()

### 4.7  Bifurcation Diagram

$\tau_0$ swept from 0.5 to 8.0 N·m. At each amplitude: 10 s transient + 10 s
stroboscopic sampling. Tight clustering → periodic; scattered cloud → chaotic.

> **⏳ This cell may take 2–4 minutes.**  
> The notebook uses 30 amplitudes and $\Delta t=4\times10^{-3}$ s for speed
> (the `.py` script uses 60 amplitudes and $\Delta t=2\times10^{-3}$ s).

In [ ]:
tau_vals = np.linspace(0.5, 8.0, 30)   # 30 amplitudes (full script uses 60)
dt_bif   = DT * 4                       # coarser step for speed
t_trans  = 10.0
t_samp   = 10.0

fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)

for tau_val in tau_vals:
    tau0 = tau_val
    t_r, Y_r = integrate_rk4(derivs, y0, (0, t_trans+t_samp), dt_bif)
    st  = np.arange(t_trans, t_trans+t_samp, T_drive)
    idx = np.searchsorted(t_r, st)
    idx = idx[idx < len(t_r)]
    for i in range(4):
        th_m = np.mod(Y_r[idx,i] + np.pi, 2*np.pi) - np.pi
        axes[i].scatter([tau_val]*len(idx), np.degrees(th_m),
                        s=0.4, color='black', alpha=0.4)

tau0 = 2.0   # restore default

for i, th_l in enumerate(THETA_LABELS):
    axes[i].set_ylabel(f'{th_l} (deg)')
    axes[i].grid(alpha=0.2)
axes[-1].set_xlabel('τ₀ (N·m)')
axes[0].set_title('Bifurcation Diagram')
fig.tight_layout()
fig.savefig(PLOTS / 'bifurcation.png', dpi=150)
plt.show()

### 4.8  Animation (Inline HTML5)

The cell below renders an inline HTML5 video of the quadruple pendulum over the
first 10 s at 25 fps. Frame selection uses `np.searchsorted` (binary search /
bisection from the Roots lecture) — no interpolation library needed.

The same function in `quadruple_pendulum.py` (lines 387–459) saves a GIF to
`plots/animation.gif` using `PillowWriter`.

In [ ]:
fps_nb     = 25
t_end_anim = 10.0
dt_frame   = 1.0 / fps_nb

# Build frame indices via binary search (nearest-neighbour)
frame_indices = []
k = 0
while k * dt_frame <= min(t_end_anim, t[-1]):
    frame_indices.append(int(np.searchsorted(t, k * dt_frame)))
    k += 1
frame_indices = np.array(frame_indices)

L_total = L1+L2+L3+L4
fig_a, ax_a = plt.subplots(figsize=(5, 6))
ax_a.set_xlim(-L_total*1.1,  L_total*1.1)
ax_a.set_ylim(-L_total*1.1,  L_total*0.35)
ax_a.set_aspect('equal')
ax_a.grid(alpha=0.3)
ax_a.set_title('Driven Quadruple Pendulum')
ax_a.set_xlabel('x (m)'); ax_a.set_ylabel('y (m)')
ax_a.plot(0, 0, 'k^', ms=8)   # fixed pivot

rods = [ax_a.plot([], [], lw=2,  color=c)[0] for c in COLORS]
bobs = [ax_a.plot([], [], 'o',   ms=8, color=c)[0] for c in COLORS]
txt  = ax_a.text(0.02, 0.95, '', transform=ax_a.transAxes, fontsize=10)

def _init():
    for r in rods: r.set_data([], [])
    for b in bobs: b.set_data([], [])
    txt.set_text('')
    return rods + bobs + [txt]

def _update(fi):
    pts = bob_positions(Y[fi])
    for kr, (r, b) in enumerate(zip(rods, bobs)):
        r.set_data([pts[kr,0], pts[kr+1,0]], [pts[kr,1], pts[kr+1,1]])
        b.set_data([pts[kr+1,0]], [pts[kr+1,1]])
    txt.set_text(f't = {t[fi]:.2f} s')
    return rods + bobs + [txt]

anim_nb = animation.FuncAnimation(
    fig_a, _update, frames=frame_indices,
    init_func=_init, blit=True, interval=1000 // fps_nb
)
plt.close(fig_a)   # suppress static frame; HTML below shows the video
HTML(anim_nb.to_jshtml())

## 5  Reflections and Discussion

### Physical Insights

- **Sensitivity to initial conditions:** Even with 64-bit floating-point precision
  ($\approx16$ significant digits), trajectories diverge within tens of seconds.
  The log-scale Lyapunov plot quantifies this.

- **Energy redistribution:** The driving torque acts only on pendulum 1, but
  energy flows to pendulums 2–4 through the angle-dependent coupling in $\mathbf{M}$.
  Pendulum 4 — receiving energy through three chaotic intermediaries — shows
  the most irregular behaviour.

- **Route to chaos:** The bifurcation diagram reveals the classic period-doubling
  cascade. With four degrees of freedom the chaotic onset occurs at lower driving
  amplitude than for a triple pendulum, and the attractor structure is richer.

### Limitations

The model assumes rigid massless rods, point masses, and no dissipation.
Adding linear damping $-b\dot\theta_i$ per equation would be straightforward.
The fixed timestep ($10^{-3}$ s) may be insufficient during rapid whipping of
pendulum 4; an adaptive step-size controller would be more efficient, but the
fixed-step RK4 satisfies the project requirement of a self-implemented algorithm.

## 6  Conclusion

The driven quadruple pendulum was simulated by deriving the Lagrangian equations
of motion, reformulating them as an eight-dimensional first-order ODE system, and
integrating with a hand-coded RK4 scheme. The $4\times4$ mass-matrix linear system
is solved at each RK4 stage via LU decomposition. The simulation reproduces the
expected phenomenology: irregular time series, space-filling phase portraits,
scattered Poincaré sections, exponential sensitivity to initial conditions,
and a bifurcation diagram showing the route from periodicity to chaos.
An inline animation illustrates the qualitative motion. The custom RK4 integrator
agrees with scipy's adaptive solver to within $10^{-6}$ rad over 20 s.

## 7  References

1. J. M. T. Thompson and H. B. Stewart, *Nonlinear Dynamics and Chaos*, 2nd ed. (Wiley, 2002).
2. S. H. Strogatz, *Nonlinear Dynamics and Chaos*, 2nd ed. (Westview Press, 2015).
3. W. H. Press et al., *Numerical Recipes*, 3rd ed. (Cambridge University Press, 2007).
4. PC3236 Computational Methods in Physics, Project Handout, NUS (2025/26).

## Declaration on the Use of Generative AI

I declare that I **HAVE** used generative AI tools to produce this assignment.

| AI Tool | Prompt / Output | How Used |
|---|---|---|
| Claude | Code structure for quadruple pendulum simulation + debugging guidance | Reference for organising code. All equations derived independently and verified against textbooks. RK4 from lecture notes. Sign errors corrected manually via energy-conservation testing. |
| Claude | Typst and Jupyter formatting for figures, tables, equations | Formatting only. All physical analysis and report content written and verified by me. |